In [ ]:
## Deep Neural Network for MNIST Classification

## The MNIST dataset is a collection of handwritten digits, and it's widely used to train algorithms for
## pattern recognition. The dataset consists on 70,000 28x28px images of handwritten digits (1 digit per
## image). The goal of this project is to write an algorithm that detects which digit is written. Since 
## there are only 10 digits (0, 1, 2, 3, 4, 5, 6, 7, 8, 9), this is a classification problem with 10 
## classes. Our goal would be to build a neural network with 2 hidden layers.

## SETUP

## Import the relevant packages

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
## LOADING AND PREPROCESSING DATA

## Download the MNIST dataset
## With_info=True will provide a tuple containing information about the version, features and 
## number of samples. Since we will use this information, we will store it in mnist_info.
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

## Extract the training and testing dataset with the built references.
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

## By default, TF has training and testing datasets, but no validation sets.
## Define the number of validation samples as a % of the train samples.
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples

## Cast this number to an integer, as a float may cause an error along the way.
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

## Create a dedicated variable to store the number of test samples.
num_test_samples = mnist_info.splits['test'].num_examples

## Once more, we'd prefer an integer.
num_test_samples = tf.cast(num_test_samples, tf.int64)

## Normally, data would be scaled in a way that would make the result more numerically stable. In
## this case we will simply prefer to have inputs between 0 and 1. Let's define a function
## called: scale, that will take an MNIST image and its label.
def scale(image, label):
    ## We make sure the value is a float.
    image = tf.cast(image, tf.float32)
    ## Since the possible values for the inputs are 0 to 255 (256 different shades of grey), if we 
    ## divide each element by 255, we would get the desired result -> all elements will be between 0 and 1. 
    image /= 255.
    return image, label

## Then, we map the scale function to the training and testing datasets.
## This will apply the scale function to each element of the dataset.
scaled_train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

## Shuffling the data

## We set a buffer size for shuffling the data. This is a parameter that will determine how 
## many samples are stored in memory at a time for shuffling.
BUFFER_SIZE = 10000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

## The validation data would be equal to 10% of the training set, which we've already calculated.
## We use the .take() method to take that many samples and then we create a batch with a batch 
## size equal to the total number of validation samples. Same thing with the training data, but 
## we skip the validation samples.
validation_data = shuffled_train_and_validation_data.take(num_validation_samples)
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

## Determine the batch size
BATCH_SIZE = 100

## Batching the training data (helpful for training the model, as it will iterate through different batches)
train_data = train_data.batch(BATCH_SIZE)
validation_data = validation_data.batch(num_validation_samples)

# Batch the test data
test_data = test_data.batch(num_test_samples)

validation_inputs, validation_targets = next(iter(validation_data))


In [ ]:
## OUTLINING THE MODEL

input_size = 784
output_size = 10
hidden_layer_size = 50
    
## Define how the model will look like
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)), # Input layer
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'), # 1st hidden layer
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'), # 2nd hidden layer
    tf.keras.layers.Dense(output_size, activation='softmax') # Output layer
])

## Input layer ---> 28x28px images, which is 784 pixels in total. We flatten the input to a 1D array of 784 elements.
## 1st hidden layer ---> 50 neurons, with ReLU activation function.
## 2nd hidden layer ---> 50 neurons, with ReLU activation function.
## Output layer ---> 10 neurons, with softmax activation function. Each neuron will represent a digit (0-9) and the
## Softmax function will output a probability distribution over the 10 classes.

## CHOOSING THE OPTIMIZER AND LOSS FUNCTION

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
## TRAINING THE MODEL

## Determine the maximum number of epochs
NUM_EPOCHS = 5

## We fit the model, specifying the training data, the total number of epochs and the validation data we just 
## created ourselves in the format (inputs,targets)
model.fit(train_data, epochs=NUM_EPOCHS, validation_data=(validation_inputs, validation_targets), verbose =2)

In [ ]:
## TEST THE MODEL

test_loss, test_accuracy = model.evaluate(test_data)
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

## The final test accuracy is around 97.5%, which is a good result for a simple neural network with 2 hidden layers
## and 50 neurons each.

In [ ]:
##